## Assessment Overview & Architecture
This notebook addresses **Task 2: SQL Database + Python Data Analysis** 

### Table Schema
The analysis operates on an existing MySQL table defined as follows:
```sql
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    order_date DATE,
    amount DECIMAL(10,2),
    status VARCHAR(20)
);
```
*(Status examples include `'Completed'`, `'Pending'`, and `'Cancelled'`.)*


## Required Python Environment Setup
Before running this notebook, ensure the required packages are installed in your Python environment:
```bash
pip install pandas mysql-connector-python python-dotenv
```


## Cell 1 — Imports, Database Configuration, and Connection Setup

### Objective
- Import `pandas`, `mysql.connector`, and supporting libraries.
- Configure database credentials securely via environment variables or a `.env` file (with sensible local defaults).
- Establish a single, persistent MySQL connection.
- Define a reusable helper function `run_query(sql, params=None)` that executes parameterized queries and returns clean Pandas DataFrames while safely closing cursors after each execution.


In [3]:
!pip install pandas mysql-connector-python python-dotenv

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB 163.8 kB/s eta 0:01:48
   ---------------------------------------- 0.0/17.7 MB 163.8 kB/s eta 0:01:48
   ---------------------------------------- 0.0/17.7 MB 163.8 kB/s eta 0:01:48
   ---------------------------------------- 0.0/17.7 MB 115.9 kB/s eta 0:02:33
   ---------------------------------------- 0.0/17.7 MB 115.9 kB/s eta 0:02:33
   ---------------------------------------- 0.1/17.7 MB 149.3 kB/s eta 0:01:59
   ---------------------------------------- 0.1/17.7 MB 149.3 kB/s eta 0:01:59
   ------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\ambre\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import sys
import pandas as pd
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv


load_dotenv()

# 2. Database Configuration.
DB_CONFIG = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'port': int(os.getenv('MYSQL_PORT', 3306)),
    'user': os.getenv('MYSQL_USER', 'root'),
    'password': os.getenv('MYSQL_PASSWORD', 'sejal@21'),  
    'database': os.getenv('MYSQL_DATABASE', 'sales_db')            

# 3. Establish Persistent MySQL Connection 
try:
    connection = mysql.connector.connect(
        host=DB_CONFIG['host'],
        port=DB_CONFIG['port'],
        user=DB_CONFIG['user'],
        password=DB_CONFIG['password'],
        database=DB_CONFIG['database'],
        autocommit=True
    )
    if connection.is_connected():
        db_info = connection.get_server_info()
        print(f"✅ Successfully connected to MySQL Server version {db_info}")
        print(f"📦 Connected to Database: '{DB_CONFIG['database']}' on {DB_CONFIG['host']}:{DB_CONFIG['port']}")
except Error as e:
    connection = None
    print(f"❌ Connection Error: Failed to connect to MySQL database.")
    print(f"   Details: {e}")
    print("   Tip: Verify host, port, user, password, database name, and ensure the MySQL server is running.")

# 4. Reusable Query Execution Helper
def run_query(sql: str, params=None) -> pd.DataFrame:
    """
    Executes an SQL query using the existing active MySQL connection and returns
    the results as a Pandas DataFrame.
    
    Parameters:
        sql (str): Parameterized SQL query string.
        params (tuple or dict, optional): Parameters to bind to the query.
        
    Returns:
        pd.DataFrame: Query results formatted as a DataFrame.
    """
    global connection
    if connection is None or not connection.is_connected():
        raise ConnectionError("No active MySQL connection found. Please verify credentials and re-run Cell 1.")
    
    cursor = None
    try:
        cursor = connection.cursor()
        cursor.execute(sql, params or ())
        
        # If the query returns rows 
        if cursor.description:
            columns = [col[0] for col in cursor.description]
            records = cursor.fetchall()
            df = pd.DataFrame(records, columns=columns)
            return df
        else:
            return pd.DataFrame()
            
    except Error as err:
        print(f"⚠️ SQL Execution Error: {err}")
        return pd.DataFrame()
        
    finally:
        
        if cursor is not None:
            cursor.close()


✅ Successfully connected to MySQL Server version 8.0.46
📦 Connected to Database: 'sales_db' on localhost:3306


C:\Users\ambre\AppData\Local\Temp\ipykernel_15400\1873440163.py:32: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_info = connection.get_server_info()


In [ ]:
# Cell 1b (Optional): Automated Sample Data Population
POPULATE_SAMPLE_DATA = True

if POPULATE_SAMPLE_DATA:
    if connection is None or not connection.is_connected():
        print("❌ Cannot populate data: No active MySQL connection. Run Cell 1 first.")
    else:
        cursor = None
        try:
            cursor = connection.cursor()
            
            # Step 1: Ensure orders table exists according to assessment schema
            create_table_sql = """
            CREATE TABLE IF NOT EXISTS orders (
                order_id INT PRIMARY KEY,
                customer_id INT,
                order_date DATE,
                amount DECIMAL(10,2),
                status VARCHAR(20)
            );
            """
            cursor.execute(create_table_sql)
            
            # Step 2: Define structured sample data covering all assessment test scenarios:
            
            sample_orders = [
                (1001, 1, '2024-01-15', 250.00, 'Completed'),
                (1002, 2, '2024-01-20', 120.50, 'Completed'),
                (1003, 1, '2024-02-05', 450.00, 'Completed'),
                (1004, 3, '2024-02-14', 310.00, 'Completed'),
                (1005, 4, '2024-02-28', 180.00, 'Pending'),
                (1006, 2, '2024-03-02', 650.00, 'Completed'),
                (1007, 5, '2024-03-10', 90.00,  'Cancelled'),
                (1008, 3, '2024-03-18', 215.50, 'Completed'),
                (1009, 1, '2024-04-01', 500.00, 'Completed'),
                (1010, 6, '2024-04-12', 340.00, 'Pending'),
                (1011, 4, '2024-04-20', 420.00, 'Completed'),
                (1012, 7, '2024-04-25', None,   'Completed'), 
                (1013, 8, '2024-05-02', 0.00,   'Completed'),  
                (1014, 2, '2024-05-10', -75.00, 'Completed'),  
                (1015, 5, '2024-05-15', 150.00, 'Completed')
            ]
            
            # Step 3: Insert or update records idempotently
            insert_sql = """
            INSERT INTO orders (order_id, customer_id, order_date, amount, status)
            VALUES (%s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                customer_id = VALUES(customer_id),
                order_date  = VALUES(order_date),
                amount      = VALUES(amount),
                status      = VALUES(status);
            """
            cursor.executemany(insert_sql, sample_orders)
            connection.commit()
            
            print(f"✅ Successfully seeded/updated {len(sample_orders)} sample records in 'orders' table.")
            
            # Display current row count in table
            cursor.execute("SELECT COUNT(*) FROM orders;")
            total_count = cursor.fetchone()[0]
            print(f"📊 Current total records in 'orders' table: {total_count}")
            
        except Error as err:
            print(f"⚠️ Error while populating sample data: {err}")
        finally:
            if cursor is not None:
                cursor.close()
else:
    print("ℹ️ Sample data population skipped (POPULATE_SAMPLE_DATA is False).")

✅ Successfully seeded/updated 15 sample records in 'orders' table.
📊 Current total records in 'orders' table: 15


## Cell 2 — Question 1: Total Completed Sales

### Problem Statement
Write and execute an SQL query to calculate the total sales value for orders whose status is `Completed`.

### Implementation & Accounting Logic
- Filter condition: `status = 'Completed'`.
- Data Quality Rule: In accordance with standard financial integrity principles (and our findings in Question 4), we ensure only positive sales transactions (`amount > 0`) are aggregated.
- Aggregate function: `SUM(amount)` with an alias `total_completed_sales`.
- Format: Returned directly as a Pandas DataFrame.


In [ ]:
# Question 1: Total completed sales
query_q1 = """
SELECT 
    COUNT(*) AS completed_order_count,
    ROUND(COALESCE(SUM(amount), 0), 2) AS total_completed_sales
FROM orders
WHERE status = %s
  AND amount > 0;
"""

params_q1 = ('Completed',)
df_q1 = run_query(query_q1, params_q1)

print("--- Question 1: Total Completed Sales ---")
display(df_q1)


--- Question 1: Total Completed Sales ---


,completed_order_count,total_completed_sales
0,9,3066.00


## Cell 3 — Question 2: Top 3 Customers by Completed Order Value

### Problem Statement
Write and execute an SQL query to find the top three customers based on the total value of their completed orders.

### Requirements
Return:
- `customer_id`
- `completed_order_value`
- `completed_order_count`
- Sorted by `completed_order_value` in descending order.
- Limited to the top 3 records.


In [12]:
# Question 2: Top 3 customers by completed order value
query_q2 = """
SELECT 
    customer_id,
    ROUND(SUM(amount), 2) AS completed_order_value,
    COUNT(order_id) AS completed_order_count
FROM orders
WHERE status = %s
  AND amount > 0
GROUP BY customer_id
ORDER BY completed_order_value DESC
LIMIT 3;
"""

params_q2 = ('Completed',)
df_q2 = run_query(query_q2, params_q2)

print("--- Question 2: Top 3 Customers ---")
display(df_q2)


--- Question 2: Top 3 Customers ---


,customer_id,completed_order_value,completed_order_count
0,1,1200.00,3
1,2,770.50,2
2,3,525.50,2


## Cell 4 — Question 3: Order Counts by Status

### Problem Statement
Write and execute an SQL query to count the number of orders grouped by their status.

### Requirements
- Include all statuses present in the `orders` table.
- Group by `status`.
- Sort results clearly (e.g., descending order count, or alphabetically).


In [13]:
# Question 3: Order counts by status
query_q3 = """
SELECT 
    status,
    COUNT(*) AS order_count,
    ROUND(COALESCE(SUM(amount), 0), 2) AS total_status_amount
FROM orders
GROUP BY status
ORDER BY order_count DESC;
"""

df_q3 = run_query(query_q3)

print("--- Question 3: Order Counts by Status ---")
display(df_q3)


--- Question 3: Order Counts by Status ---


,status,order_count,total_status_amount
0,Completed,12,2991.00
1,Pending,2,520.00
2,Cancelled,1,90.00


## Cell 5 — Question 4: Data Quality Issues

### Problem Statement
Write and execute an SQL query to identify orders with data quality anomalies:
- Missing (`NULL`) amounts
- Zero (`0.00`) amounts
- Negative amounts (`< 0.00`)

### Requirements
- Return `order_id`, `customer_id`, `order_date`, `amount`, `status`, and a calculated `issue` column detailing the problem.
- A `CASE` statement categorizes each anomaly clearly for triage and auditing.


In [14]:
# Question 4: Data quality issues identification
query_q4 = """
SELECT 
    order_id,
    customer_id,
    order_date,
    amount,
    status,
    CASE 
        WHEN amount IS NULL THEN 'Missing (NULL) Amount'
        WHEN amount = 0 THEN 'Zero Amount'
        WHEN amount < 0 THEN 'Negative Amount'
        ELSE 'Valid'
    END AS issue
FROM orders
WHERE amount IS NULL 
   OR amount <= 0
ORDER BY order_date DESC, order_id ASC;
"""

df_q4 = run_query(query_q4)

print("--- Question 4: Data Quality Audit Report ---")
if df_q4.empty:
    print("✅ No data quality issues (NULL, zero, or negative amounts) found in the orders table.")
else:
    print(f"⚠️ Found {len(df_q4)} order(s) with data quality anomalies:")
    display(df_q4)


--- Question 4: Data Quality Audit Report ---
⚠️ Found 3 order(s) with data quality anomalies:


,order_id,customer_id,order_date,amount,status,issue
0,1014,2,2024-05-10,-75.00,Completed,Negative Amount
1,1013,8,2024-05-02,0.00,Completed,Zero Amount
2,1012,7,2024-04-25,None,Completed,Missing (NULL) Amount


## Cell 6 — Question 5: Monthly Completed Sales Using Pandas

### Problem Statement
Use Python and Pandas to analyze monthly completed sales trends:
1. Fetch completed orders from MySQL using the existing connection.
2. Load the query result into a Pandas DataFrame.
3. Convert `order_date` to a `datetime` type.
4. Filter out invalid/non-positive amounts to maintain data integrity.
5. Group orders by month and calculate total completed sales per month.
6. Sort results chronologically.
7. Display the monthly summary report.
8. Save the output to `monthly_completed_sales.csv`.

### Data Quality & Accounting Rule Explanation
- **Policy:** In sales revenue reporting, only positive amounts (`amount > 0`) represent realized gross sales.
- **Missing (`NULL`) Amounts:** Excluded because missing values cannot be verified as financial transactions.
- **Zero (`0.00`) Amounts:** Excluded from sales totals as they provide no revenue contribution.
- **Negative Amounts:** Excluded from gross sales totals because negative amounts represent returns, refunds, or bookkeeping corrections, which should be audited separately rather than mixed into gross sales.


In [15]:
# Question 5: Monthly completed sales using Pandas
# Step 1: Fetch all completed orders from MySQL using the active connection
query_q5 = """
SELECT 
    order_id,
    customer_id,
    order_date,
    amount,
    status
FROM orders
WHERE status = %s;
"""

params_q5 = ('Completed',)
df_completed = run_query(query_q5, params_q5)

print(f"Raw completed orders fetched: {len(df_completed)}")

if df_completed.empty:
    print("No completed orders found in database.")
else:
    # Step 2: Ensure order_date is converted to datetime
    df_completed['order_date'] = pd.to_datetime(df_completed['order_date'], errors='coerce')
    
    # Step 3: Apply consistent data quality rule (exclude NULL, zero, and negative amounts)
    valid_sales_mask = (df_completed['amount'].notna()) & (df_completed['amount'] > 0)
    df_valid = df_completed[valid_sales_mask].copy()
    
    excluded_count = len(df_completed) - len(df_valid)
    if excluded_count > 0:
        print(f"ℹ️ Excluded {excluded_count} record(s) with missing, zero, or negative amounts based on accounting integrity rules.")
    
    # Step 4: Extract Month in Year-Month format for grouping and sorting
    df_valid['year_month'] = df_valid['order_date'].dt.to_period('M')
    
    # Step 5: Group by month and compute aggregates (Total Sales & Order Count)
    monthly_sales = (
        df_valid.groupby('year_month')
        .agg(
            total_completed_sales=('amount', 'sum'),
            completed_order_count=('order_id', 'count')
        )
        .reset_index()
    )
    
    # Step 6: Sort chronologically and format columns
    monthly_sales = monthly_sales.sort_values('year_month', ascending=True)
    monthly_sales['year_month'] = monthly_sales['year_month'].astype(str)
    monthly_sales['total_completed_sales'] = monthly_sales['total_completed_sales'].round(2)
    
    # Step 7: Display the monthly report
    print("--- Question 5: Monthly Completed Sales Report ---")
    display(monthly_sales)
    
    # Step 8: Save the output to monthly_completed_sales.csv
    csv_filename = "monthly_completed_sales.csv"
    monthly_sales.to_csv(csv_filename, index=False)
    print(f"💾 Successfully exported monthly sales summary to '{csv_filename}'.")


Raw completed orders fetched: 12
ℹ️ Excluded 3 record(s) with missing, zero, or negative amounts based on accounting integrity rules.
--- Question 5: Monthly Completed Sales Report ---


,year_month,total_completed_sales,completed_order_count
0,2024-01,370.50,2
1,2024-02,760.00,2
2,2024-03,865.50,2
3,2024-04,920.00,2
4,2024-05,150.00,1


💾 Successfully exported monthly sales summary to 'monthly_completed_sales.csv'.


## Cell 7 — Question 6: Avoiding Double-Counting in SQL Joins

### Problem Statement
Explain how to avoid double-counting orders when joining `orders` with another table containing multiple records per customer (e.g., customer addresses, order line items, support tickets, or reward points).

---

### 1. The Root Cause: Cardinality Explosion (Fan-Out)
When an `orders` table is joined to a related table where the join key matches **multiple rows** (a 1-to-many relationship, such as `customers 1 ──< customer_addresses` or `customers 1 ──< support_tickets`), the SQL join duplicates the rows of the `orders` table for every matching row in the second table.

#### Problematic Example:
Imagine Customer 101 placed **one order** for $100.00, but has **three shipping addresses** registered:
```sql
-- ❌ DANGEROUS QUERY: Duplicates order rows and inflates sales totals!
SELECT 
    o.customer_id,
    SUM(o.amount) AS total_sales
FROM orders o
JOIN customer_addresses a ON o.customer_id = a.customer_id
WHERE o.status = 'Completed'
GROUP BY o.customer_id;
```
**Result of Fan-Out:** The single $100.00 order row is matched 3 times (once per address). `SUM(o.amount)` reports **$300.00**, artificially tripling the customer's sales volume!

---

### 2. Why `SUM(DISTINCT amount)` Is NOT a Reliable General Solution
A common beginner instinct is to patch the query using `SUM(DISTINCT o.amount)`. **This approach is fundamentally flawed and dangerous in production:**

1. **Collapses Legitimate Duplicate Values:** If a customer legitimately placed two separate, genuine $50.00 orders on different dates, `DISTINCT o.amount` discards one of the $50.00 values because the *monetary amounts are identical*. The true total of $100.00 is erroneously halved to $50.00.
2. **Fails on Recurring Transactions:** In subscription services, recurring fee models, or flat-rate retail pricing, identical order amounts are standard. `SUM(DISTINCT amount)` completely distorts financial reporting.
3. **Ignores Multi-Order Cardinality:** `DISTINCT` operates on values, not on transaction entities (`order_id`).

---

### 3. The Correct Solutions: Pre-Aggregation and CTEs
To eliminate row multiplication, **always aggregate data at the correct grain before joining**, or join only on unique keys.

#### Method A: Pre-Aggregate Orders First (Customer-Level Rollup)
Summarize sales at the customer level *before* joining to customer attributes:
```sql
-- ✅ SOLUTION A: Pre-aggregate orders before joining to customer attributes
WITH customer_sales AS (
    SELECT 
        customer_id,
        SUM(amount) AS total_completed_sales,
        COUNT(order_id) AS total_orders
    FROM orders
    WHERE status = 'Completed' AND amount > 0
    GROUP BY customer_id
)
SELECT 
    c.customer_id,
    c.customer_name,
    COALESCE(cs.total_completed_sales, 0) AS total_completed_sales,
    COALESCE(cs.total_orders, 0) AS total_orders
FROM customers c
LEFT JOIN customer_sales cs ON c.customer_id = cs.customer_id;
```

#### Method B: Pre-Aggregate / Deduplicate the Dimension Table
If joining to a child table (e.g., shipping addresses or tags), aggregate or pick the primary record *before* the join:
```sql
-- ✅ SOLUTION B: Deduplicate child table before joining
WITH primary_address AS (
    SELECT customer_id, address_line, city
    FROM customer_addresses
    WHERE is_primary = 1  -- Ensure strictly 1 row per customer
)
SELECT 
    o.customer_id,
    pa.city,
    SUM(o.amount) AS total_sales
FROM orders o
LEFT JOIN primary_address pa ON o.customer_id = pa.customer_id
WHERE o.status = 'Completed' AND o.amount > 0
GROUP BY o.customer_id, pa.city;
```

*(The executable cell below prints a concise summary of these architectural principles for quick reference during assessment demonstrations.)*


In [ ]:
# Question 6: Architectural demonstration of avoiding join multiplication


explanation_summary = """
================================================================================
💡 KEY TAKEAWAYS: AVOIDING DOUBLE-COUNTING IN SQL JOINS
================================================================================
1. The Problem (Row Multiplication):
   - Joining a 1-to-many relationship (e.g., Orders to Addresses/Tickets) duplicates
     order rows for each secondary record, artificially inflating SUM(amount).

2. Why SUM(DISTINCT amount) Fails:
   - It deduplicates by dollar value, not transaction ID. If a customer legitimately
     places two orders of $50 each, SUM(DISTINCT) computes $50 instead of $100!

3. The Industry-Standard Solution:
   - Pre-aggregate the data at the appropriate granular entity level using CTEs 
     (Common Table Expressions) or derived subqueries BEFORE performing the JOIN.
================================================================================
"""
print(explanation_summary)



💡 KEY TAKEAWAYS: AVOIDING DOUBLE-COUNTING IN SQL JOINS
1. The Problem (Row Multiplication):
   - Joining a 1-to-many relationship (e.g., Orders to Addresses/Tickets) duplicates
     order rows for each secondary record, artificially inflating SUM(amount).

2. Why SUM(DISTINCT amount) Fails:
   - It deduplicates by dollar value, not transaction ID. If a customer legitimately
     places two orders of $50 each, SUM(DISTINCT) computes $50 instead of $100!

3. The Industry-Standard Solution:
   - Pre-aggregate the data at the appropriate granular entity level using CTEs 
     (Common Table Expressions) or derived subqueries BEFORE performing the JOIN.



## Cell 8 — Optional Data Preview (First 20 Orders)

### Objective
Inspect the raw structure and sample rows of the `orders` table using the existing database connection.


In [17]:
# Optional: Preview the first 20 records from the orders table
query_preview = """
SELECT 
    order_id,
    customer_id,
    order_date,
    amount,
    status
FROM orders
LIMIT 20;
"""

df_preview = run_query(query_preview)

print(f"--- Orders Table Preview (First {len(df_preview)} Rows) ---")
display(df_preview)


--- Orders Table Preview (First 15 Rows) ---


,order_id,customer_id,order_date,amount,status
0,1001,1,2024-01-15,250.00,Completed
1,1002,2,2024-01-20,120.50,Completed
2,1003,1,2024-02-05,450.00,Completed
3,1004,3,2024-02-14,310.00,Completed
4,1005,4,2024-02-28,180.00,Pending
5,1006,2,2024-03-02,650.00,Completed
6,1007,5,2024-03-10,90.00,Cancelled
7,1008,3,2024-03-18,215.50,Completed
8,1009,1,2024-04-01,500.00,Completed
9,1010,6,2024-04-12,340.00,Pending


## Cell 9 — Close Database Connection

### Objective
Safely release database resources by closing the persistent connection once all analysis cells have finished executing.


In [18]:
# Cell 9: Safe Connection Closure
if 'connection' in globals() and connection is not None:
    try:
        if connection.is_connected():
            connection.close()
            print("🔒 MySQL database connection successfully closed.")
        else:
            print("ℹ️ MySQL connection was already closed.")
    except Exception as e:
        print(f"⚠️ Error while closing connection: {e}")
else:
    print("ℹ️ No active MySQL connection found to close.")


🔒 MySQL database connection successfully closed.
